# 第 6 週 實作｜部分分式分解

積分技巧最後一招。有理函數看起來最無害卻積不動——要先把它拆開。而「拆」這件事骨子裡是在解一組線性方程組:這是線性代數的第一個伏筆。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜手刻部分分式分解器:解一個線性系統

觀念 7 說「比較係數＝解 $M\mathbf{c}=\mathbf{b}$」。這格把它寫成程式——十行 numpy 取代所有手算,也是你的第一支線性代數程式。


In [ ]:
x = sp.Symbol('x')

def decompose_distinct_linear(numer_coeffs, roots):
    """把 P(x) / prod(x - r_i) 拆開(相異一次因式)。

    做法:設 P(x)/prod = sum_i A_i/(x - r_i),清分母後比較係數,
    得線性系統 M c = b,用 numpy 解。
    numer_coeffs: 分子係數,由高次到低次
    roots: 分母各一次因式的根
    """
    n = len(roots)
    # 第 i 個未知數對應的多項式 = prod_{j != i} (x - r_j)
    basis = []
    for i in range(n):
        p = np.poly1d([1.0])
        for j, r in enumerate(roots):
            if j != i:
                p = p * np.poly1d([1.0, -r])
        basis.append(np.pad(p.coefficients, (n - 1 - len(p.coefficients) + 1, 0)))
    M = np.array(basis).T                      # 每一行是一個基底多項式的係數
    b = np.pad(np.array(numer_coeffs, dtype=float),
               (M.shape[0] - len(numer_coeffs), 0))
    return np.linalg.solve(M, b)

# 例 1: (3x + 5) / ((x-1)(x+2))
coef = decompose_distinct_linear([3, 5], [1, -2])
print("(3x+5)/((x-1)(x+2)) 的係數:", np.round(coef, 6), "  理論值 [8/3, 1/3] =",
      np.round([8/3, 1/3], 6))

# 例 2: 1 / ((x-1)(x-4))
coef2 = decompose_distinct_linear([5], [1, 4])
print("5/((x-1)(x-4))     的係數:", np.round(coef2, 6), "  理論值 [-5/3, 5/3]")

# 用 sympy 交叉驗證
print("\nSymPy apart() 的答案:")
print("  ", sp.apart((3*x + 5)/((x - 1)*(x + 2)), x))
print("  ", sp.apart(5/((x - 1)*(x - 4)), x))

In [ ]:
# TODO 學生練習:用這支程式解 (2x - 1) / ((x+1)(x-2)) 的係數
# 再手算驗證(遮蓋法):x=-1 代入 (2x-1)/(x-2),x=2 代入 (2x-1)/(x+1)

## Lab 2｜四招決策樹:讓程式幫你選

期中考前的整理。把本學期學過的四招寫成一張表,用 SymPy 檢查每一題的答案,並看清楚「哪一題該用哪一招」。


In [ ]:
x = sp.Symbol('x', real=True)

problems = [
    ("∫ x·sqrt(1-x²) dx",   x*sp.sqrt(1-x**2),          "換元 t=1-x²"),
    ("∫ sqrt(1-x²) dx",     sp.sqrt(1-x**2),            "三角代換 x=sinθ"),
    ("∫ x·ln x dx",         x*sp.log(x),                "分部 (LIATE: u=ln x)"),
    ("∫ x·e^x dx",          x*sp.exp(x),                "分部 (u=x)"),
    ("∫ dx/(x²-1)",         1/(x**2-1),                 "部分分式"),
    ("∫ dx/(x²+1)",         1/(x**2+1),                 "標準公式 → arctan"),
    ("∫ (2x+1)/(x²+x) dx",  (2*x+1)/(x**2+x),           "換元(分子=分母導數!)"),
    ("∫ sin³x·cos²x dx",    sp.sin(x)**3*sp.cos(x)**2,  "奇次拆一個"),
]

print(f"{'積分':24s} {'建議招式':26s} {'SymPy 的答案'}")
for name, f, method in problems:
    r = sp.simplify(sp.integrate(f, x))
    print(f"{name:24s} {method:26s} {str(r)[:44]}")

print("\n注意第 7 題:分子 2x+1 恰好是分母 x²+x 的導數 —— 不必拆,直接換元。")
print("      拿到有理函數先看這個,可以省一大段。")

In [ ]:
# TODO 學生練習:再加三題到 problems 裡,分別對應
# (1) 需要先長除的假分式 (2) 需要先配方的 (3) 含不可約二次因式的
# 跑跑看 SymPy 給什麼答案,和你手算的對不對得上